In [1]:
# ============================================================================
# PROJECT SETUP
# ============================================================================

import sys
from pathlib import Path

import sqlite3
import pandas as pd

PROJECT_ROOT = Path.cwd().parents[1]

if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

from src.config import (
    DATABASE_FILE,
)

# Phase 4 Validation Workbook

## Data Quality Framework

### Objectif

Évaluer la qualité des données stockées dans la base SQLite du projet avant leur utilisation dans les phases suivantes.

Les contrôles réalisés portent sur :

- les valeurs manquantes ;
- les doublons ;
- les formats incorrects ;
- les valeurs aberrantes ;
- la cohérence des devises ;
- la chronologie des données.

## Database Validation

### Objectif

Vérifier la disponibilité de la base SQLite utilisée par le framework qualité.

In [2]:
DATABASE_FILE.exists()

True

In [3]:
conn = sqlite3.connect(
    DATABASE_FILE
)

pd.read_sql(
    """
    SELECT name
    FROM sqlite_master
    WHERE type='table'
    """,
    conn,
)

,name
0,security_candidates
1,securities_master
2,securities_unmatched


## Security Master

### Objectif

Charger la table principale utilisée par le framework qualité.

In [4]:
securities_master = pd.read_sql(
    """
    SELECT *
    FROM securities_master
    """,
    conn,
)

securities_master.shape

(120, 17)

In [5]:
securities_master.head()

,ticker,name_source,location,exchange_source,currency,asset_class,exchange_code,figi,composite_figi,share_class_figi,security_name,security_type,security_type_2,market_sector,security_description,match_count,status
0,6669,WIWYNN CORPORATION,Taiwan,Taiwan Stock Exchange,USD,Equity,TT (Taiwan Stock Exchange),BBG00J4ZF054,BBG00J4ZF036,BBG00J4ZF090,WIWYNN CORP,Common Stock,Common Stock,Equity,6669,1,MATCH
1,NVDA,NVIDIA,United States,NASDAQ,USD,Equity,US,BBG000BBJQV0,BBG000BBJQV0,BBG001S5TZJ6,NVIDIA CORP,Common Stock,Common Stock,Equity,NVDA,1,MATCH
2,BMY,BRISTOL MYERS SQUIBB,United States,NYSE,USD,Equity,US,BBG000DQLV23,BBG000DQLV23,BBG001S8N8J6,BRISTOL-MYERS SQUIBB CO,Common Stock,Common Stock,Equity,BMY,1,MATCH
3,VWS,VESTAS WIND SYSTEMS,Denmark,Omx Nordic Exchange Copenhagen A/S,USD,Equity,DC,BBG000BJBK53,BBG000BJBJM7,BBG001S7TVH3,VESTAS WIND SYSTEMS A/S,Common Stock,Common Stock,Equity,VWS,1,MATCH
4,TSN,TYSON FOODS INC CLASS A,United States,NYSE,USD,Equity,US,BBG000DKCC19,BBG000DKCC19,BBG001S871D5,TYSON FOODS INC-CL A,Common Stock,Common Stock,Equity,TSN,1,MATCH


## Missing Values

### Objectif

Identifier les valeurs manquantes présentes dans le Security Master.

In [6]:
securities_master.isna().sum()

ticker                   0
name_source              0
location                 0
exchange_source          0
currency                 0
asset_class              0
exchange_code            0
figi                    17
composite_figi          17
share_class_figi        17
security_name           17
security_type           17
security_type_2         17
market_sector           17
security_description    17
match_count              0
status                   0
dtype: int64

## Duplicates

### Objectif

Identifier les doublons potentiels.

In [7]:
securities_master.duplicated().sum()

np.int64(0)

In [8]:
securities_master[
    securities_master.duplicated(
        keep=False
    )
]

,ticker,name_source,location,exchange_source,currency,asset_class,exchange_code,figi,composite_figi,share_class_figi,security_name,security_type,security_type_2,market_sector,security_description,match_count,status


## Invalid Formats

### Objectif

Vérifier que les statuts observés appartiennent aux valeurs autorisées.

In [10]:
securities_master[
    "status"
].value_counts()

status
MATCH       103
NO_MATCH     17
Name: count, dtype: int64

## Outliers

### Objectif

Détecter les incohérences évidentes dans les données disponibles.

In [11]:
securities_master[
    [
        "ticker",
        "match_count",
        "status",
    ]
]

,ticker,match_count,status
0,6669,1,MATCH
1,NVDA,1,MATCH
2,BMY,1,MATCH
3,VWS,1,MATCH
4,TSN,1,MATCH
...,...,...,...
115,COLPAL,0,NO_MATCH
116,3692,1,MATCH
117,605117,1,MATCH
118,688187,1,MATCH


In [12]:
securities_master[
    (securities_master["status"] == "MATCH")
    &
    (
        securities_master["match_count"] <= 0
    )
]

,ticker,name_source,location,exchange_source,currency,asset_class,exchange_code,figi,composite_figi,share_class_figi,security_name,security_type,security_type_2,market_sector,security_description,match_count,status
